In [ ]:
# Load necessary libraries
install.packages(c("dplyr", "tidyr"))


In [1]:
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
# Read the data; make sure the name in the file upload matches the name in the read_csv bubble
# Do not change the skeleton
data <- read.csv("Perceptual_FullMoonDataRaw090225.csv")

In [ ]:
# optional print for debugging
print(data)

In [3]:
# Function to calculate visual angle
calc_va_df <- function(df, size_col, dist_col) {
  size <- df[[size_col]]
  dist <- df[[dist_col]]
  (2 * atan(size / (2 * dist))) * (180 / pi)
}

# Calculate Visual Angle of All Objects
data <- data %>%
  mutate(
  # Visual Angles produced by probes
    Lower_Elevation_Perceptual_VA  = calc_va_df(data, "Round.1.Estimate.Template.Size..cm.", "Round.1.Estimate.Distance..cm."),
    Higher_Elevation_Perceptual_VA  = calc_va_df(data, "Round.2.Estimate.Template.Size..cm.", "Round.2.Estimate.Distance..cm."),
# Visual Angle of the Moon
    Real_Visual_Angle = calc_va_df(data, "Moon.Diameter..km.", "Moon.Distance..km."),
# Disparity Visual Angle
    Diameter = data$Moon.Diameter..km.,
    Distance = data$Moon.Distance..km.,
    Lower_Elevation = data$Elevation.1..deg.,
    Higher_Elevation = data$Elevation.2..deg.,
    Lower_Time = data$`Time.1`,
    Higher_Time = data$`Time.2`)

data_long <- data %>%
  select(ID = 1, Gender, Date, Lower_Time, Higher_Time, Age, Distance,
        Lower_Elevation_Perceptual_VA,
        Higher_Elevation_Perceptual_VA, Real_Visual_Angle,
        Lower_Elevation, Higher_Elevation) %>%
  pivot_longer(
    cols = c(Lower_Elevation_Perceptual_VA, Higher_Elevation_Perceptual_VA),
    names_to = "Measurement",
    values_to = "Reported_Visual_Angle") %>%
  filter(!is.na(Reported_Visual_Angle)) %>%
  mutate(

  Time = case_when(
      grepl("Lower_Elevation", Measurement) ~ Lower_Time,
      grepl("Higher_Elevation", Measurement) ~ Higher_Time
    ),
    Task = case_when(
      grepl("Perceptual", Measurement) ~ "Perceptual",
    ),
    Ratio_Visual_Angle = Reported_Visual_Angle / Real_Visual_Angle,

    Elevation = case_when(
      grepl("Lower_Elevation", Measurement) ~ Lower_Elevation,
      grepl("Higher_Elevation", Measurement) ~ Higher_Elevation
    ),
    Session = case_when(
      grepl("Lower_Elevation", Measurement) ~ "Lower",
      grepl("Higher_Elevation", Measurement) ~ "Higher"
    ),

    Template_Distance = case_when(
      grepl("Lower_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.1.Estimate.Distance..cm.[match(ID, data$ID)],
      grepl("Higher_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.2.Estimate.Distance..cm.[match(ID, data$ID)],
      TRUE ~ NA_real_
  ),
  Template_Size = case_when(
      grepl("Lower_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.1.Estimate.Template.Size..cm.[match(ID, data$ID)],
      grepl("Higher_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.2.Estimate.Template.Size..cm.[match(ID, data$ID)],
      TRUE ~ NA_real_
  )
  )

data_long <- data_long %>% select(-Higher_Elevation, -Lower_Elevation, -Measurement, -Lower_Time, -Higher_Time)
data_long <- filter(data_long)


In [4]:
write.csv(data_long, file = "Perceptual_FullMoonDataProcessed090225.csv", row.names = FALSE)